In [6]:
import sys
sys.path.append("/Users/sujaladhikari/Desktop/FedIDS")

In [7]:
import os 
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from FederatedLearningModel.fedmodel import MLP
from torch.optim import Adam

In [ ]:
dataset = pd.read_csv('../silos_datasets/combined_binary_silos.csv')
dataset = dataset.drop(columns = 'Unnamed: 0')
dataset.head(5)

In [ ]:
print(dataset['Label_Binary'].value_counts())
print(f"The dataset has following number {dataset.shape[0]} number of rows and {dataset.shape[1]} number of columns")

Label_Binary
0    556488
1    556488
Name: count, dtype: int64
The dataset has following number 1112976 number of rows and 79 number of columns


In [ ]:
random_seed = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
X = dataset.drop(columns = 'Label_Binary')
X = X.to_numpy()
y = dataset['Label_Binary']
y = y.to_numpy()


In [ ]:
## Spliting the data into train, test, and validation splits
X_train , X_vals, y_train, y_vals = train_test_split(X,y , test_size=0.3, random_state=random_seed)
X_val, X_test, y_val, y_test = train_test_split(X_vals, y_vals, test_size=0.5, random_state=random_seed)

In [ ]:
### Standard Scaling for all the datas

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.long)

X_val = scaler.transform(X_val)
X_val = torch.tensor(X_val, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.long)

X_test = scaler.transform(X_test)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.long)



In [ ]:
training_data = TensorDataset(X_train,y_train)
testing_data = TensorDataset(X_test, y_test)
validation_data = TensorDataset(X_val, y_val)


In [ ]:
training_batch = DataLoader(training_data, batch_size = 64, shuffle = True)
testing_batch = DataLoader(training_data, batch_size=64, shuffle=False)
validation_batch = DataLoader(validation_data,batch_size = 64, shuffle = False)

In [ ]:
print(MLP)

<class 'FederatedLearningModel.fedmodel.MLP'>


In [ ]:
input_size = dataset.shape[0]
hidden_layer = [256, 128,64,8]
num_classes = 2

In [ ]:
## We will be using the same model that is being used in fedmodel.py for the federated purpose
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(input_size, hidden_layer, num_classes).to(device)
### Optimizer and loss function 
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
loss = nn.CrossEntropyLoss()
num_epoch = 10
model = model.to(device)

NameError: name 'torch' is not defined

In [ ]:
### Training, testing and evaluation of the centralized model 
## setting up the training environment:
def train(model, train_loader,validation_loader, criterion, device):
    model.train() ## Starting to train the model 
    training_loss = 0.0 ## Calculating the training loss in terms of floating value
    train_correct = 0 ## Count of total number of correct samples in total 
    training_total = 0 ## Count of the total number of samples checked
    for samples, classes in train_loader: ## Loading the data loader
        samples = samples.to(device) ## Converting the sample and classes to the device ('cpu')
        classes = classes.to(device)
        prediciton = model(samples) ## Prediction for the samples in the batch 
        optimizer.zero_grad() ## Started the calculation of the loss 
        loss = criterion(prediciton, classes)
        loss.backward() ## Backward propagation 
        optimizer.step() ## Forward optimizatio 
        training_loss +=loss.item() * samples.size(0) ## Calculating the loss for each batch 
        _,predicted = prediciton.max(1) ## Max prediction for each sample in the batch 
        training_total += classes.size(0) ### Each total in training adds up 
        train_correct += predicted.eq(classes).sum().item() ## Converting the total predicted that are equal to the classes and sum them 
    training_loss = training_loss/len(train_loader.dataset)
    training_accuracy = 100 * train_correct/ training_total 
    val_loss = 0
    val_correct = 0
    val_total = 0 
    model.eval()
    with torch.no_grad(): ## Evaluation started
        for val_batch in validation_loader:
            features, labels = val_batch 
            features = features.to(device) ## Converted the path to device 'cpu'
            labels = labels.to(device)
            val_outputs = model(features) ## Outputs
            val_loss_batch = criterion(val_outputs, labels) ## loss calculated
            val_loss += val_loss_batch.item() * features.size(0) ## for each batch 
            _, val_predicted = val_outputs.max(1) ## Maximum value calcuated
            val_correct += val_predicted.eq(labels).sum().item()
            val_total += labels.size(0) 
    avg_validation_loss = val_loss / len(validation_loader.dataset) ## Average across the sum 
    validation_accuracy = 100 * val_correct/ val_total ## Accuracy 
    return training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct

In [ ]:
### Creating the evalution function 
def evaluate(model, test_loader, critertion, device):
    model.eval()
    test_loss = 0.0 ## This is for the test loss 
    correct = 0 
    total = 0 
    predictions = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            samples, labels = batch 
            samples = samples.to(device)
            labels = samples.to(labels)
            outputs = model(samples)
            loss = critertion(outputs, labels)
            _, predicted = outputs.max(1)
            test_loss += loss.item() * samples.size(0)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            predictions.extend(predicted.tolist())
            true_labels.extend(predicted.tolist())
        
    test_loss = test_loss / len(test_loader.dataset)
    accuracy = 100 * correct / total 

    return test_loss, accuracy, predictions, true_labels

### Since the training process has been done, we will evaluate the model's perfomance 

In [ ]:
### On the given epoch we will be training and testing the epcoh 
num_epoch
for epoch in range(num_epoch):
    training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct = train(model, training_batch,validation_batch, loss, device)
    test_loss, test_accuracy, predictions, true_labels = evaluate(model, testing_batch, loss, device)
    print(f"Epoch [{epoch+1}/{num_epoch}], Training Loss: {training_loss:4f}, Testing Loss: {test_loss:4f}")
    

NameError: name 'num_epoch' is not defined